### Phase 1: Chronic Kidney Disease Dataset Inspection

In [2]:
%pip install scikit-learn==1.9.0
import sklearn
print(sklearn.__version__)

1.9.0


In [3]:
import pandas as pd
import numpy as np

# 1. Load the actual uploaded NHANES CKD dataset
df = pd.read_csv('/content/CKD_NHANES_2021_2023.csv')

# 2. Basic Shape and Columns Info
print("=== Dataset Shape ===")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")

print("=== Column Types ===")
print(df.dtypes)
print("\n")

# 3. Head Preview
print("=== First 5 Rows ===")
display(df.head())

# 4. Target Column Distribution
# Let's inspect potential target column or display columns so we can identify it
print("=== Available Columns in this Dataset ===")
print(df.columns.tolist())


=== Dataset Shape ===
Rows: 11933, Columns: 29

=== Column Types ===
participant_id              float64
age                         float64
gender                       object
ethnicity                    object
education_level             float64
poverty_income_ratio        float64
bmi                         float64
weight_kg                   float64
height_cm                   float64
bp_systolic                 float64
bp_diastolic                float64
serum_creatinine            float64
blood_urea_nitrogen         float64
albumin_serum               float64
phosphorus                  float64
bicarbonate                 float64
calcium                     float64
uric_acid                   float64
urine_creatinine            float64
urine_albumin               float64
albumin_creatinine_ratio    float64
diabetes_diagnosed          float64
insulin_use                 float64
diabetes_pills              float64
ever_smoked                 float64
current_smoker              flo

,participant_id,age,gender,ethnicity,education_level,poverty_income_ratio,bmi,weight_kg,height_cm,bp_systolic,...,urine_albumin,albumin_creatinine_ratio,diabetes_diagnosed,insulin_use,diabetes_pills,ever_smoked,current_smoker,egfr,ckd_stage,ckd_present
0,130378.0,43.0,Male,Non-Hispanic Asian,5.0,5.00,27.0,86.9,179.5,135.0,...,23.12,17.00,2.0,NaN,NaN,1.0,3.0,112.61,No CKD,0
1,130379.0,66.0,Male,Non-Hispanic White,5.0,5.00,33.5,101.8,174.2,121.0,...,4.25,6.64,2.0,NaN,NaN,1.0,3.0,97.98,No CKD,0
2,130380.0,44.0,Female,Other Hispanic,3.0,1.41,29.7,69.4,152.9,111.0,...,12.43,7.92,1.0,2.0,1.0,2.0,NaN,111.69,No CKD,0
3,130381.0,5.0,Female,Other/Multiracial,NaN,1.53,23.8,34.3,120.1,NaN,...,16.12,7.75,2.0,NaN,NaN,NaN,NaN,NaN,Unknown,1
4,130382.0,2.0,Male,Non-Hispanic White,NaN,3.60,NaN,13.6,NaN,NaN,...,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,Unknown,1


=== Available Columns in this Dataset ===
['participant_id', 'age', 'gender', 'ethnicity', 'education_level', 'poverty_income_ratio', 'bmi', 'weight_kg', 'height_cm', 'bp_systolic', 'bp_diastolic', 'serum_creatinine', 'blood_urea_nitrogen', 'albumin_serum', 'phosphorus', 'bicarbonate', 'calcium', 'uric_acid', 'urine_creatinine', 'urine_albumin', 'albumin_creatinine_ratio', 'diabetes_diagnosed', 'insulin_use', 'diabetes_pills', 'ever_smoked', 'current_smoker', 'egfr', 'ckd_stage', 'ckd_present']


In [4]:
# 5. Analyze Missing Values
missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_count, 'Percentage (%)': missing_percent})
print("=== Missing Values Summary ===")
display(missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False))

# 6. Check for Duplicate Rows
duplicate_rows = df.duplicated().sum()
print(f"\nNumber of absolute duplicate rows: {duplicate_rows}")

# 7. Check target class distribution
print("\n=== Target Class ('ckd_present') Distribution ===")
print(df['ckd_present'].value_counts(dropna=False))


=== Missing Values Summary ===


,Missing Count,Percentage (%)
insulin_use,10852,90.941088
diabetes_pills,9652,80.884941
current_smoker,8690,72.823263
phosphorus,5611,47.020867
bicarbonate,5608,46.995726
egfr,5607,46.987346
blood_urea_nitrogen,5607,46.987346
serum_creatinine,5607,46.987346
uric_acid,5604,46.962206
calcium,5571,46.685662



Number of absolute duplicate rows: 0

=== Target Class ('ckd_present') Distribution ===
ckd_present
1    8341
0    3592
Name: count, dtype: int64


In [5]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

# 1. Load the verified dataset
df = pd.read_csv('/content/CKD_NHANES_2021_2023.csv')

target_col = 'ckd_present'

if target_col not in df.columns:
    raise ValueError(
        f"Target '{target_col}' not found. Actual columns: {df.columns.tolist()}"
    )

# Remove rows with a missing target; do not impute labels
df = df.dropna(subset=[target_col]).copy()

# Confirm this is binary 0/1 classification before continuing
print("Target distribution:")
print(df[target_col].value_counts(dropna=False))

if set(df[target_col].unique()) != {0, 1}:
    raise ValueError(
        "Expected target labels 0 and 1. Verify the dataset target mapping first."
    )

# 2. Exclude identifiers, target-derived columns, and very sparse fields.
# Confirm these exclusions against the dataset documentation before final use.
columns_to_drop = [
    'participant_id',
    'egfr',
    'ckd_stage',
    'insulin_use',
    'diabetes_pills',
    'current_smoker'
]

X = df.drop(
    columns=[target_col] + columns_to_drop,
    errors='ignore'
).copy()

y = df[target_col].astype(int)

# Treat numeric-coded categories as categorical rather than continuous numbers.
categorical_candidates = [
    'gender',
    'ethnicity',
    'education_level',
    'diabetes_diagnosed',
    'ever_smoked'
]

categorical_cols = [
    col for col in categorical_candidates if col in X.columns
]

# Any remaining numeric columns are numerical features
numerical_cols = [
    col for col in X.select_dtypes(include=np.number).columns
    if col not in categorical_cols
]

# If other non-numeric columns exist, include them as categorical features
other_categorical = [
    col for col in X.columns
    if col not in numerical_cols and col not in categorical_cols
]
categorical_cols = categorical_cols + other_categorical

print("\nNumerical features:", numerical_cols)
print("\nCategorical features:", categorical_cols)
print("\nFeature count:", X.shape[1])

# 3. Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 4. Candidate models
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        random_state=42
    )
}

# 5. Compare models using training-set cross-validation only
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
best_model_name = None
best_cv_f1 = -1
best_pipeline = None

for name, classifier in models.items():

    # Build a fresh preprocessor for each candidate model
    numerical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', classifier)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='f1',
        n_jobs=-1
    )

    mean_cv_f1 = scores.mean()
    cv_results[name] = mean_cv_f1

    print(f'{name} - 5-fold CV F1: {mean_cv_f1:.4f}')

    if mean_cv_f1 > best_cv_f1:
        best_cv_f1 = mean_cv_f1
        best_model_name = name
        best_pipeline = pipeline

print(f'\nSelected model using CV only: {best_model_name}')
print(f'Mean CV F1: {best_cv_f1:.4f}')

# 6. Fit selected Pipeline on the full training split
best_pipeline.fit(X_train, y_train)

# 7. Evaluate once on the untouched test split
y_pred = best_pipeline.predict(X_test)

print('\n=== Final Test Set Evaluation ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred, zero_division=0):.4f}')
print(f'F1-score:  {f1_score(y_test, y_pred, zero_division=0):.4f}')

print('\nClassification report:')
print(classification_report(y_test, y_pred, zero_division=0))

print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

# ROC-AUC when the fitted classifier supports probabilities
if hasattr(best_pipeline, 'predict_proba'):
    class_labels = list(best_pipeline.classes_)
    positive_index = class_labels.index(1)
    y_pred_proba = best_pipeline.predict_proba(X_test)[:, positive_index]
    print(f'ROC-AUC:   {roc_auc_score(y_test, y_pred_proba):.4f}')
else:
    y_pred_proba = None
    print('ROC-AUC skipped: selected estimator has no predict_proba.')

# Preserve these variables for the artifact-saving cell
print('\nPipeline training completed. Review data leakage and label definition before deployment.')



Target distribution:
ckd_present
1    8341
0    3592
Name: count, dtype: int64

Numerical features: ['age', 'poverty_income_ratio', 'bmi', 'weight_kg', 'height_cm', 'bp_systolic', 'bp_diastolic', 'serum_creatinine', 'blood_urea_nitrogen', 'albumin_serum', 'phosphorus', 'bicarbonate', 'calcium', 'uric_acid', 'urine_creatinine', 'urine_albumin', 'albumin_creatinine_ratio']

Categorical features: ['gender', 'ethnicity', 'education_level', 'diabetes_diagnosed', 'ever_smoked']

Feature count: 22
Logistic Regression - 5-fold CV F1: 0.9092
Random Forest - 5-fold CV F1: 0.9812
Gradient Boosting - 5-fold CV F1: 0.9857

Selected model using CV only: Gradient Boosting
Mean CV F1: 0.9857

=== Final Test Set Evaluation ===
Accuracy:  0.9820
Precision: 0.9909
Recall:    0.9832
F1-score:  0.9871

Classification report:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       719
           1       0.99      0.98      0.99      1668

    accuracy        

In [6]:
import joblib
import json
from google.colab import files

# 1. Save the entire fitted pipeline
pipeline_filename = 'kidney_disease_pipeline.pkl'
joblib.dump(best_pipeline, pipeline_filename)
print(f"Fitted pipeline saved successfully to: {pipeline_filename}")

# 2. Build metadata consistent with Liver Disease module
kidney_metadata = {
  "disease": "kidney_disease",
  "model_file": "kidney_disease_pipeline.pkl",
  "pipeline_type": "scikit-learn Pipeline",
  "selected_model": best_model_name,
  "dataset_file": "CKD_NHANES_2021_2023.csv",
  "target_column": "ckd_present",
  "features": X_train.columns.tolist(),
  "target_mapping": {
    "0": "No chronic kidney disease (ckd_present=0)",
    "1": "Chronic kidney disease present (ckd_present=1)"
  },
  "test_metrics": {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred)),
    "recall": float(recall_score(y_test, y_pred)),
    "f1_score": float(f1_score(y_test, y_pred)),
    "roc_auc": float(roc_auc_score(y_test, y_pred_proba))
  }
}

metadata_filename = 'kidney_disease_metadata.json'
with open(metadata_filename, 'w') as f:
    json.dump(kidney_metadata, f, indent=2)

print(f"Metadata saved successfully to: {metadata_filename}")

# Trigger browser downloads for both files
print("Downloading files to your local system...")
files.download(pipeline_filename)
files.download(metadata_filename)


Fitted pipeline saved successfully to: kidney_disease_pipeline.pkl
Metadata saved successfully to: kidney_disease_metadata.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>